# ESM2 Protein Kaggle Runner

Purpose: run Project 15 only in an approved Kaggle session, preserving separation between source checks, synthetic validation, evidence capture, and gated real data/model stages.

Safety rules: do not download ProteinGym or model checkpoints unless approval is recorded; do not call providers; do not upload to W&B or Hugging Face; do not run overnight jobs; do not delete historical evidence; do not publish raw rows, sequences, embeddings, checkpoints, hidden predictions, caches, or fitted artifacts.

Expected outputs: command output, structured skipped/failed/completed records, and sanitized evidence summaries under `/kaggle/working/esm2-protein/evidence/`.

In [ ]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

print('python:', sys.version)
print('platform:', platform.platform())
print('cwd:', Path.cwd())
print('kaggle working exists:', Path('/kaggle/working').exists())
print('kaggle input exists:', Path('/kaggle/input').exists())
print('cuda visible devices:', os.environ.get('CUDA_VISIBLE_DEVICES', '<unset>'))
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
input_root = Path('/kaggle/input')
if not input_root.exists():
    print('No /kaggle/input directory visible.')
else:
    for path in sorted(input_root.iterdir()):
        print(path)

In [ ]:
import shutil

source_candidates = [Path('/kaggle/input/esm2-protein-fitness'), Path('/kaggle/input/project-15-esm2-protein'), Path.cwd()]
source_root = next((path for path in source_candidates if (path / 'pyproject.toml').exists()), None)
if source_root is None:
    raise FileNotFoundError('Attach or upload the esm2-protein-fitness source tree; no pyproject.toml found.')
working_root = Path('/kaggle/working/esm2-protein')
if working_root.exists():
    raise FileExistsError(f'{working_root} already exists; inspect it manually before overwriting.')
ignore = shutil.ignore_patterns('.git', '.venv', '__pycache__', '.pytest_cache', 'artifacts_restricted', 'data_restricted', 'checkpoints', 'embeddings', 'caches', 'hidden_predictions', 'fitted_artifacts', 'wandb')
shutil.copytree(source_root, working_root, ignore=ignore)
print('copied source_root:', source_root)
print('working_root:', working_root)

In [ ]:
working_root = Path('/kaggle/working/esm2-protein')
%cd /kaggle/working/esm2-protein
# Kaggle-only install. Run only if pytest/imports are missing in the current kernel.
!{sys.executable} -m pip install -e . pytest

## Kernel Restart Gate

After dependency installation, restart the Kaggle kernel from the menu. Then rerun the purpose, environment, input inspection, and source-copy cells. If the source directory already exists after restart, inspect it manually before continuing.

In [ ]:
%cd /kaggle/working/esm2-protein
env = dict(os.environ)
env['PYTHONPATH'] = str(Path.cwd() / 'src')
commands = [
    [sys.executable, '-m', 'compileall', 'src', 'tests'],
    [sys.executable, '-m', 'pytest', '-q'],
    [sys.executable, '-m', 'esm2_fitness.pipeline', 'check'],
    [sys.executable, '-m', 'esm2_fitness.pipeline', 'synthetic'],
    [sys.executable, '-m', 'esm2_fitness.pipeline', 'gates'],
]
for command in commands:
    print('\n$', ' '.join(command))
    completed = subprocess.run(command, env=env, text=True, capture_output=True, check=False)
    print('exit:', completed.returncode)
    print(completed.stdout)
    print(completed.stderr)
    if completed.returncode != 0:
        raise SystemExit(f'command failed: {command}')

In [ ]:
evidence_root = Path('/kaggle/working/esm2-protein/evidence')
evidence_root.mkdir(parents=True, exist_ok=True)
existing = sorted(evidence_root.rglob('*'))
print('evidence files:')
for path in existing:
    if path.is_file():
        print(path.relative_to(evidence_root), path.stat().st_size)

## Approval Gate

Stop here unless explicit approval is recorded for real data, model checkpoints, providers, GPU training, or heavy CPU work. Approval must name the dataset/model paths, permitted commands, resource class, and output boundary. If approval is missing, write a skipped evidence summary and end the session.

In [ ]:
APPROVED_REAL_DATA = False
APPROVED_MODEL_DOWNLOADS = False
APPROVED_GPU_OR_HEAVY_CPU = False
approved_paths = []

approval = {
    'real_data': APPROVED_REAL_DATA,
    'model_downloads': APPROVED_MODEL_DOWNLOADS,
    'gpu_or_heavy_cpu': APPROVED_GPU_OR_HEAVY_CPU,
    'approved_paths': approved_paths,
}
print(json.dumps(approval, indent=2))
if not all([APPROVED_REAL_DATA, APPROVED_MODEL_DOWNLOADS, APPROVED_GPU_OR_HEAVY_CPU]):
    raise SystemExit('Approval gate closed. Record skipped evidence and stop.')

In [ ]:
# Optional approved command 1: gate real-data path only. Set APPROVED_DATA_DIR before running.
APPROVED_DATA_DIR = ''
command = [sys.executable, '-m', 'esm2_fitness.pipeline', 'real', '--data-dir', APPROVED_DATA_DIR, '--allow-external-data']
completed = subprocess.run(command, env=env, text=True, capture_output=True, check=False)
print('exit:', completed.returncode)
print(completed.stdout)
print(completed.stderr)

In [ ]:
# Optional approved command 2: run a future ESM1v parity command only after it exists and is approved.
# Keep one command per cell and preserve stdout/stderr in evidence.
raise SystemExit('No approved ESM1v runtime command is implemented in the local source tree yet.')

In [ ]:
# Optional approved command 3: run a future frozen ESM2 smoke command only after it exists and is approved.
# Keep model assets under approved Kaggle input/working paths and do not publish embeddings.
raise SystemExit('No approved ESM2 runtime command is implemented in the local source tree yet.')

In [ ]:
summary = {
    'project': 'esm2-protein',
    'source_root': str(Path('/kaggle/working/esm2-protein')),
    'synthetic_validation': 'run above before completing this summary',
    'real_data': 'not_run_without_approval',
    'model_stages': 'not_run_without_approval',
    'restricted_artifacts_publication': 'prohibited',
}
summary_path = Path('/kaggle/working/esm2-protein/evidence/final_sanitized_summary.json')
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(summary_path)
print(json.dumps(summary, indent=2, sort_keys=True))
raise SystemExit('Stop. Preserve evidence and paste sanitized summary into the handoff.')